# AUPE Processing for Ries Crater Field Trip May 2025

A notebook for developing a processing library and pipeline for AUPE colour and multispectral image processing.

In this notebook we provide a batch processing pipeline to process all of the data from the campaign.

Developed for processing of the Ries Crater field trip, May 2025.

R. Stabbins  
Natural History Museum, London, UK  
7/5/2025

# Overview

# Notebook and Scene Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'svg' # note - requires Inkscape to be installed, and to be accessible from the terminal
# on macos, add inkscape to the path with `sudo ln -s /Applications/Inkscape.app/Contents/MacOS/inkscape /usr/local/bin/inkscape`
%matplotlib inline

In [ ]:
import aupy

We're going to be looking at the AUPE scene that contains the ColorChecker in each of the WACs and HRC.

In [ ]:
from pathlib import Path

In [ ]:
data_dir = Path('..', 'data')
# check that the data directory exists
if not data_dir.exists():
    raise FileNotFoundError(f"Data directory {data_dir} does not exist.")

In [ ]:
# make a list of all the directories in the data directory
sols = [d.name for d in data_dir.iterdir()]
sols.sort()

In [ ]:
sols = sols[1:]

# Batch Script

In [ ]:
for sol in sols:
    # get the scenes in the sol directory
    sol_dir = data_dir / sol
    # check that the sol directory exists
    if not sol_dir.exists():
        raise FileNotFoundError(f"Sol directory {sol_dir} does not exist.")
    # get the scenes in the sol directory
    scenes = [d.name for d in sol_dir.iterdir() if d.is_dir()]
    scenes.sort()
    
    print(f'{sol}: {scenes}')
    for scene in scenes:

        # look for subdirectorys in the scene directory
        scene_dir = sol_dir / scene
        # check that the scene directory exists
        if not scene_dir.exists():
            raise FileNotFoundError(f"Scene directory {scene_dir} does not exist.")
        # get the subdirectories in the scene directory
        trials = [d.name for d in scene_dir.iterdir() if d.is_dir()]
        trials.sort()

        if trials == []:
            trials = ['']

        print(f'  {sol} {scene}: {trials}')      

        for trial in trials:
            print(f'    {sol} {scene} {trial}')

            # process LWAC
            lwac_rgb_msc_loader = aupy.AupeIO('LWAC', 'MSC', sol, scene, trial=trial, filter_ids=['L1R', 'L2G', 'L3B'])
            if lwac_rgb_msc_loader.input_files != []:
                # RGB Reflectance and Colour Calibration
                lwac_rgb_msc = lwac_rgb_msc_loader.load_frame()
                lwac_rgb_msc.exposure_correct()
                lwac_rgb_msc.set_false_color({'R':'L1R', 'G':'L2G', 'B':'L3B'})
                lwac_rgb_msc.false_rgb.export_image('99b')
                lwac_cal_targ = aupy.CalibrationTarget()
                targ_found = lwac_cal_targ.find_target_outline(lwac_rgb_msc.false_rgb.get_image('99b'), show=True)
                if not targ_found:
                    lwac_cal_targ.draw_target_outline(lwac_rgb_msc.get_image('99b'), show=True)
                lwac_cal_targ.get_bad_patches(lwac_rgb_msc)
                lwac_cal_targ.calibrate_reflectance(lwac_rgb_msc,show=True)
                lwac_cal_targ.get_bad_patches(lwac_rgb_msc.false_rgb)
                lwac_cal_targ.calibrate_colour(lwac_rgb_msc.false_rgb, show=True)
                lwac_rgb_msc.false_rgb.export_image('ccm')
                lwac_rgb_msc.export_2_envi()
                # Narrowband Reflectance Calibration
                lwac_msc_loader = aupy.AupeIO('LWAC', 'MSC', sol, scene, trial=trial)
                lwac_msc = lwac_msc_loader.load_frame()
                lwac_msc.exposure_correct()
                lwac_cal_targ.get_bad_patches(lwac_msc)
                print(lwac_msc.cwls)
                lwac_cal_targ.calibrate_reflectance(lwac_msc, show=True)
                lwac_msc.set_false_color({'R':'G06', 'G':'G03', 'B':'G01'})
                lwac_msc.false_rgb.export_image('99b')
                lwac_msc.export_2_envi()
                lwac_msc.export_2_gif()

            # process RWAC
            rwac_rgb_msc_loader = aupy.AupeIO('RWAC', 'MSC', sol, scene, trial=trial, filter_ids=['R1R', 'R2G', 'R3B'])
            if rwac_rgb_msc_loader.input_files != []:
                # RGB Reflectance and Colour Calibration
                rwac_rgb_msc = rwac_rgb_msc_loader.load_frame()
                rwac_rgb_msc.exposure_correct()
                rwac_rgb_msc.set_false_color({'R':'R1R', 'G':'R2G', 'B':'R3B'})
                rwac_rgb_msc.false_rgb.export_image('99b')
                rwac_cal_targ = aupy.CalibrationTarget()
                targ_found = rwac_cal_targ.find_target_outline(rwac_rgb_msc.false_rgb.get_image('99b'), show=True)
                if not targ_found:
                    rwac_cal_targ.draw_target_outline(rwac_rgb_msc.get_image('99b'), show=True)
                rwac_cal_targ.get_bad_patches(rwac_rgb_msc)
                rwac_cal_targ.calibrate_reflectance(rwac_rgb_msc,show=True)
                rwac_cal_targ.get_bad_patches(rwac_rgb_msc.false_rgb)
                rwac_cal_targ.calibrate_colour(rwac_rgb_msc.false_rgb, show=True)
                rwac_rgb_msc.false_rgb.export_image('ccm')
                rwac_rgb_msc.export_2_envi()
                # Narrowband Reflectance Calibration
                rwac_msc_loader = aupy.AupeIO('RWAC', 'MSC', sol, scene, trial=trial)
                rwac_msc = rwac_msc_loader.load_frame()
                rwac_msc.exposure_correct()
                rwac_cal_targ.get_bad_patches(rwac_msc)
                rwac_cal_targ.calibrate_reflectance(rwac_msc,show=True)
                rwac_msc.set_false_color({'R':'G11', 'G':'G09', 'B':'G07'})
                rwac_msc.false_rgb.export_image('99b')
                rwac_msc.export_2_envi()
                rwac_msc.export_2_gif()

            # warp LWAC to RWAC
            if lwac_rgb_msc_loader.input_files != [] and rwac_rgb_msc_loader.input_files != []:
                st_lrwac = aupy.StereoTools(lwac_rgb_msc.false_rgb.get_image('ccm'), rwac_rgb_msc.false_rgb.get_image('ccm'))
                st_lrwac.select_match_regions()
                # prompt user for tag
                tag = input('Enter tag for LWAC to RWAC warp: ')
                st_lrwac.findWarp(show=True)
                lrwac_msc_loader = aupy.AupeIO('LRWAC', 'MSC', sol, scene, trial=trial)
                lrwac_msc = lrwac_msc_loader.load_frame()
                st_lrwac.applyWarp(lwac_msc, rwac_msc, lrwac_msc, tag)
                lrwac_msc.set_false_color({'R': 'G11', 'G': 'G05', 'B': 'G01'})
                lrwac_msc.false_rgb.export_image('99b')
                lrwac_msc.export_2_envi()
                lrwac_msc.export_2_gif()

            # process HRC
            hrc_loader = aupy.AupeIO('HRC', 'MSC', sol, scene, trial=trial)
            hrc_rgb_loader = aupy.AupeIO('HRC', 'RGB', sol, scene, trial=trial)
            if hrc_loader.input_files != []:     
                hrc_rgb = hrc_rgb_loader.load_frame()
                hrc_rgb.export_image('99b')       
                hrc_msc = hrc_loader.load_frame()                
                hrc_msc.exposure_correct()
                hrc_msc.load_reflectance_coefficients_from_transfer(rwac_rgb_msc)
                hrc_msc.apply_reflectance_calibration()
                hrc_msc.false_rgb.export_image('ccm')
                hrc_msc.export_2_envi()

                # warp LRWAC to HRC
                if lwac_rgb_msc_loader.input_files != [] and rwac_rgb_msc_loader.input_files != [] and hrc_loader.input_files != []:
                    st_hrc = aupy.StereoTools(rwac_rgb_msc.false_rgb.get_image('ccm'), hrc_msc.false_rgb.get_image('ccm'))
                    st_hrc.select_match_regions()
                    # prompt user for tag
                    tag = input('Enter tag for LRWAC to HRC warp: ')
                    st_hrc.findWarp(show=True)
                    lrwhrc_msc_loader = aupy.AupeIO('LRWHRC', 'MSC', sol, scene, trial=trial)
                    lrwhrc_msc = lrwhrc_msc_loader.load_frame()
                    st_hrc.applyWarp(lrwac_msc, hrc_msc, lrwhrc_msc, tag)
                    lrwhrc_msc.set_false_color({'R': 'G11', 'G': 'G05', 'B': 'G01'})
                    lrwhrc_msc.false_rgb.export_image('99b')
                    lrwhrc_msc.export_2_envi()
                    lrwhrc_msc.export_2_gif()
